# Hafta 3 — Veri ile Tanışma: pandas ve Keşifsel Veri Analizi (EDA)

**Yapay Zekâ Temelleri** dersi, 3. hafta uygulama defteri.

Veri seti: `elektrik_tuketimi.csv` — bir binanın 30 gün boyunca saatlik tüketimi (720 satır).
Sütunlar: `gun, saat, hafta_sonu, sicaklik_C, tuketim_kW`. İçinde bilerek bırakılmış **12 eksik sıcaklık** ve **1 aykırı tüketim** değeri var.

## 0. Veri dosyasını Colab'a yükleme

Dersin sayfasından `elektrik_tuketimi.csv` dosyasını indirin, sonra aşağıdaki hücreyi çalıştırıp seçin. (Sol paneldeki *Dosyalar* → *Yükle* de kullanılabilir.)

In [ ]:
import os
if not os.path.exists("elektrik_tuketimi.csv"):
    try:
        from google.colab import files
        files.upload()          # açılan pencereden elektrik_tuketimi.csv dosyasını seçin
    except ImportError:
        print("Colab dışındasınız: dosyayı bu defterle aynı klasöre koyun.")
print("Dosya var mı?", os.path.exists("elektrik_tuketimi.csv"))

## 1. Veriyi yükle ve ilk bakış

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("elektrik_tuketimi.csv")
print(df.shape)     # (satır, sütun)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

**Soru:** `describe()` çıktısında `tuketim_kW` için max değer ortalamanın kaç katı? Bu size ne düşündürüyor?

In [ ]:
print("Sütun başına eksik değer sayısı:")
print(df.isna().sum())

## 2. Seçme, filtreleme, sıralama

In [ ]:
print(df["tuketim_kW"].head(3))              # tek sütun -> Series
print(df[["saat", "tuketim_kW"]].head(3))    # birden çok sütun -> DataFrame

In [ ]:
hafta_sonu = df[df["hafta_sonu"] == 1]
mesai = df[(df["saat"] >= 8) & (df["saat"] <= 18) & (df["hafta_sonu"] == 0)]
print("Hafta sonu satır sayısı:", len(hafta_sonu))
print("Mesai saatleri ortalama tüketim:", mesai["tuketim_kW"].mean().round(1), "kW")

In [ ]:
# En yüksek 5 tüketim
df.sort_values("tuketim_kW", ascending=False).head()

Dikkat: en tepedeki değer diğerlerinin çok üstünde — sensör hatası mı, gerçek olay mı? (Sınıfta tartışın.)

## 3. Gruplama ile özet

In [ ]:
profil = df.groupby("saat")["tuketim_kW"].mean()
print(profil.round(1))

In [ ]:
df.groupby("hafta_sonu")["tuketim_kW"].agg(["mean", "median", "std", "count"]).round(1)

## 4. Tanımlayıcı istatistik: derste elle yaptığımız örnekler

### Örnek 3.1 — aykırı değerin etkisi

In [ ]:
x  = np.array([100, 110, 120, 130, 140])
x2 = np.array([100, 110, 120, 130, 950])
for v in (x, x2):
    print(f"ort={v.mean():6.1f}  medyan={np.median(v):6.1f}  std={v.std():7.2f}")

### Örnek 3.2 — Pearson korelasyonu

In [ ]:
xs = np.array([20, 24, 28, 32]); ys = np.array([80, 90, 110, 140])
pay = ((xs - xs.mean()) * (ys - ys.mean())).sum()
payda = np.sqrt(((xs - xs.mean())**2).sum() * ((ys - ys.mean())**2).sum())
print("elle  :", round(pay / payda, 3))
print("numpy :", np.corrcoef(xs, ys)[0, 1].round(3))

## 5. Eksik ve aykırı değerler

### Örnek 3.3 — IQR kuralı

In [ ]:
q1, q3 = df["tuketim_kW"].quantile([0.25, 0.75])
iqr = q3 - q1
ust = q3 + 1.5 * iqr
print(f"Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}  üst sınır={ust:.1f}")
print("Aykırı satırlar:")
print(df[df["tuketim_kW"] > ust])

In [ ]:
# IQR kuralı 950 kW ile birlikte birkaç GERÇEK sıcak-gün zirvesini (230-250 kW) de işaretler.
# Gerçek olayları atmayız; sadece fiziksel olarak imkânsız değeri (bina kapasitesi ~300 kW) çıkarırız.
temiz = df[df["tuketim_kW"] < 300].copy()
# Eksik sıcaklıkları zamana göre interpolasyonla doldur
temiz["sicaklik_C"] = temiz["sicaklik_C"].interpolate()
print(temiz.shape)
print(temiz.isna().sum())

**Deneyin:** Eksikleri `interpolate()` yerine `fillna(df['sicaklik_C'].mean())` ile doldurun. İki yöntem hangi satırlarda ne kadar farklı sonuç veriyor?

## 6. Görselleştirme ile keşif

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
temiz["tuketim_kW"].hist(bins=30, ax=ax[0]); ax[0].set_title("Tüketim dağılımı"); ax[0].set_xlabel("kW")
temiz.groupby("saat")["tuketim_kW"].mean().plot(ax=ax[1], marker="o"); ax[1].set_title("Saatlik profil"); ax[1].set_ylabel("kW")
ax[2].scatter(temiz["sicaklik_C"], temiz["tuketim_kW"], s=8); ax[2].set_title("Sıcaklık - tüketim"); ax[2].set_xlabel("°C")
plt.tight_layout(); plt.show()

In [ ]:
temiz.boxplot(column="tuketim_kW", by="hafta_sonu", figsize=(5, 3.5))
plt.suptitle(""); plt.title("Gün tipine göre tüketim (0: hafta içi, 1: hafta sonu)")
plt.show()

In [ ]:
# Korelasyon matrisi
korr = temiz[["saat", "hafta_sonu", "sicaklik_C", "tuketim_kW"]].corr()
print(korr.round(2))

plt.figure(figsize=(4.5, 3.8))
plt.imshow(korr, cmap="Blues", vmin=-1, vmax=1)
plt.colorbar(label="r")
plt.xticks(range(4), korr.columns, rotation=45); plt.yticks(range(4), korr.columns)
for i in range(4):
    for j in range(4):
        plt.text(j, i, f"{korr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.title("Korelasyon matrisi"); plt.tight_layout(); plt.show()

**Grafikten çıkarım:** Saçılım grafiğinde ilişki ~24 °C'ye kadar düz, sonra artıyor (klima yükü). Bu doğrusal olmayan ilişkiyi 5. haftada `max(T − 24, 0)` gibi bir *özellik dönüşümüyle* modele sığdıracağız.

## 7. Ölçekleme (Örnek 3.4)

In [ ]:
x = np.array([20, 24, 28, 32], dtype=float)
minmax = (x - x.min()) / (x.max() - x.min())
z = (x - x.mean()) / x.std()
print("min-max :", minmax.round(3))
print("z-skor  :", z.round(3))

from sklearn.preprocessing import StandardScaler, MinMaxScaler
print("sklearn z:", StandardScaler().fit_transform(x.reshape(-1, 1)).ravel().round(3))

In [ ]:
# Tüm veri setini ölçeklemek (4. haftadan itibaren modelden önce hep yapacağız)
ozellikler = temiz[["saat", "hafta_sonu", "sicaklik_C"]]
olcekli = StandardScaler().fit_transform(ozellikler)
print("ölçekli ortalamalar :", olcekli.mean(axis=0).round(3))
print("ölçekli std'ler     :", olcekli.std(axis=0).round(3))

## 8. Sınıf içi alıştırmalar

1. En yüksek 5 tüketim saatini (gün + saat) listeleyin (temiz veride).
2. Hafta içi ve hafta sonu için saatlik profilleri **aynı eksende** iki çizgi olarak çizin.
3. Sıcaklık–tüketim korelasyonunu sadece 12–18 saatleri için hesaplayın; tüm veriyle karşılaştırın.
4. Eksik sıcaklıkları (a) ortalama (b) interpolasyon ile doldurup farkı bir grafikte gösterin.

In [ ]:
# Alıştırma 1

In [ ]:
# Alıştırma 2

In [ ]:
# Alıştırma 3

In [ ]:
# Alıştırma 4

## 9. Ödev 2 — EDA raporu

UCI **Combined Cycle Power Plant** veri setini (AT, V, AP, RH → PE) ya da bu dersin veri setini kullanarak:

1. Veriyi tanıtın (satır/sütun, türler).
2. Eksik / aykırı değer analizi yapın ve kararınızı gerekçelendirin.
3. En az 4 grafik üretin; her grafiğin altına metin hücresinde 1–2 cümle yorum yazın.
4. Hedef değişkenle en yüksek korelasyonlu iki özelliği bulun ve **fiziksel** olarak neden mantıklı olduğunu açıklayın.

Teslim: bu defterin kopyası, *Runtime → Restart and run all* ile hatasız.

**Bonus — hazır veri seti yükleme örneği (scikit-learn):**

In [ ]:
from sklearn.datasets import fetch_california_housing
konut = fetch_california_housing(as_frame=True).frame     # internet gerekir
konut.head()